#### Messages

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.
Messages are objects that contain:
 - Role - Identifies the message type (e.g. system, user)
 - Content - Represents the actual content of the message (like text, images, audio, documents, etc.)
 - Metadata - Optional fields such as response information, message IDs, and token usage

LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called.

In LangChain, message types (roles) are represented by different message classes. The core ones are:

The main 5 message types

- HumanMessage (role: user / human): input from the user.
- AIMessage (role: assistant / ai): the model's response; can include tool_calls.
- SystemMessage (role: system): instructions that set the model's behavior or context.
- ToolMessage (role: tool): the result of a tool/function call, linked back via tool_call_id.
- FunctionMessage (role: function): older, deprecated predecessor of ToolMessage (OpenAI's legacy function calling).

Additional types

- ChatMessage: a generic message where you set any custom role string yourself.
- RemoveMessage: a special message used in LangGraph to delete a message from the state by ID (not sent to the model).

Chunk variants

For streaming, each has a chunk version: `HumanMessageChunk, AIMessageChunk, SystemMessageChunk, ToolMessageChunk, ChatMessageChunk`, and so on.

In [1]:
import os

from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-20b")

In [2]:
model.invoke("Please tell me how to properly cook basmati rice")

AIMessage(content='**How to Cook Basmati Rice (Stovetop & Rice‑Cooker Methods)**  \n\nBasmati rice is prized for its long, slender grains and fragrant aroma. With the right technique, you’ll get fluffy, separate grains every time.\n\n---\n\n## 1. Ingredients & Equipment\n\n| Item | Quantity (for 4 servings) | Notes |\n|------|---------------------------|-------|\n| Basmati rice | 2 cups (≈ 400\u202fg) | 1 cup per 2 servings |\n| Water | 3 cups (≈ 720\u202fmL) | 1.5\u202f× the rice volume |\n| Salt | ½\u202ftsp | Optional |\n| Olive oil or butter | 1\u202ftsp | Optional, for extra fluff |\n| Rice cooker or saucepan (with tight‑fitting lid) | 1 | |\n\n---\n\n## 2. Rinse & (Optional) Soak\n\n1. **Rinse**: Place rice in a fine‑mesh strainer. Rinse under cold running water until the water runs clear.  \n   *Why?* Removes excess starch that can make rice gummy.  \n\n2. **Soak** (optional but recommended for extra fluffy rice):  \n   * 20–30\u202fmin in cold water.  \n   * Drain well before c

### Text Prompts
Text prompts are strings - ideal for straightforward generation tasks where you don’t need to retain conversation history.

In [3]:
model.invoke("what is a roux")

AIMessage(content='A **roux** (pronounced “roo‑ee”) is a classic French cooking base made by cooking equal parts of fat (usually butter, but sometimes oil, bacon fat, or animal fat) and flour together. The mixture is cooked until the raw flour taste is gone and it develops a color that ranges from pale blond to deep brown, depending on how long it’s cooked.  \n\n### Why use a roux?\n- **Thickening agent**: It forms a stable emulsion that thickens sauces, soups, stews, and gravies.\n- **Flavor base**: The cooking process caramelizes the flour, adding nutty, toasty notes to the dish.\n- **Texture**: It gives a smooth, velvety mouthfeel to creamy dishes.\n\n### Basic steps\n1. **Measure**: Start with equal parts by weight (e.g., 30\u202fg butter + 30\u202fg flour). By volume, a common ratio is 1\u202fcup flour to 1\u202fcup fat, but weight gives more consistency.\n2. **Melt the fat**: In a saucepan, melt the butter over medium heat.\n3. **Add flour**: Stir in the flour and keep it moving 

Use text prompts when:
- You have a single, standalone request
- You don’t need conversation history
- You want minimal code complexity

### Message Prompts
Alternatively, you can pass in a list of messages to the model by providing a list of message objects.


Message types
- System message - Tells the model how to behave and provide context for interactions
- Human message - Represents user input and interactions with the model
- AI message - Responses generated by the model, including text content, tool calls, and metadata
- Tool message - Represents the outputs of tool calls

### System Message
A SystemMessage represent an initial set of instructions that primes the model’s behavior. You can use a system message to set the tone, define the model’s role, and establish guidelines for responses.


### Human Message
A HumanMessage represents user input and interactions. They can contain text, images, audio, files, and any other amount of multimodal content.

### AI Message
An AIMessage represents the output of a model invocation. They can include multimodal data, tool calls, and provider-specific metadata that you can later access.

### Tool Message
For models that support tool calling, AI messages can contain tool calls. Tool messages are used to pass the results of a single tool execution back to the model.

In [ ]:
from langchain.messages import SystemMessage, HumanMessage,AIMessage

messages=[
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a poem about biryani")
]

response=model.invoke(messages)
response.content

In [ ]:
system_msg = SystemMessage("You are a helpful cooking assistant.")

messages = [
    system_msg,
    HumanMessage("How do I preheat a convection oven?")
]
response = model.invoke(messages)
print(response.content)

In [ ]:
## Detailed info to the LLM through System message
from langchain.messages import SystemMessage, HumanMessage

system_msg = SystemMessage("""
You are a senior chef with expertise in kitchen equipment and techniques.
Always provide clear steps and explain your reasoning.
Be concise but thorough in your explanations.
""")

messages = [
    system_msg,
    HumanMessage("How do I preheat a convection oven?")
]
response = model.invoke(messages)
print(response.content)

In [ ]:
## Message Metadata
human_msg = HumanMessage(
    content="Hello!",
    name="ravi",  # Optional: identify different users
    id="msg_123",  # Optional: unique identifier for tracing
)

In [ ]:
response = model.invoke([
  human_msg
])
response

In [ ]:
from langchain.messages import AIMessage, SystemMessage, HumanMessage

# Create an AI message manually (e.g., for conversation history)
ai_msg = AIMessage("I'd be happy to help you with that question!")

# Add to conversation history
messages = [
    SystemMessage("You are a helpful cooking assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,  # Insert as if it came from the model
    HumanMessage("Great! What temperature should I bake cookies at?")
]

response = model.invoke(messages)
print(response.content)

In [ ]:
response.usage_metadata

In [ ]:
from langchain.messages import AIMessage
from langchain.messages import ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "check_ingredient_stock",
        "args": {"ingredient": "saffron"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
stock_result = "In stock, 15g available"
tool_message = ToolMessage(
    content=stock_result,
    tool_call_id="call_123"  # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("Is saffron in stock?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result
]
response = model.invoke(messages)  # Model processes the result

In [ ]:
tool_message

In [ ]:
response

### 🤔 Before you move on
Every response so far has been free-form text (`response.content`) that you'd have to manually parse if you wanted structured data out of it. What if you want the model to hand back a *guaranteed* schema — a Pydantic object, a TypedDict, a dataclass — instead of a string you have to pick apart yourself? That's `5-structuredoutput.ipynb`.